# Build an AI engineer agent with Claude Managed Agents

## Introduction

An AI engineer's job is to turn "we should use an LLM for this" into working, testable software: retrieval pipelines, prompt templates, and — crucially — an evaluation harness that proves the system answers correctly. In this cookbook you'll build an agent that does exactly that: hand it a knowledge-base document, get back a complete RAG (retrieval-augmented generation) project with code, tests, an eval suite, and a README.

You'll run it on [Claude Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview), Anthropic's hosted runtime for stateful, tool-using agents, built on four core concepts:

- **Agent**: the model, system prompt, tools, MCP servers, and skills
- **Environment**: a configured container template (packages, network access)
- **Session**: a running agent instance within an environment, performing a specific task and generating outputs
- **Events**: messages exchanged between your application and the agent (user turns, tool results, status updates)

### What you'll learn

By the end of this cookbook you'll be able to:

- Enable the web tools so the agent can consult current documentation while it builds
- Get a multi-file software project (not just a report) out of a session
- Use a stub LLM client pattern so the generated project's tests run offline and deterministically

### Prerequisites

- Python 3.11+
- An Anthropic API key from the [Console](https://platform.claude.com/settings/keys), set as `ANTHROPIC_API_KEY`

Install dependencies:

In [ ]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [ ]:
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv, set_key

load_dotenv()
client = Anthropic()
MODEL = "claude-sonnet-4-6"

## 1. Create an environment

The environment installs the `anthropic` SDK (so the generated project can target it), `scikit-learn` for TF-IDF retrieval, and `pytest` for the eval harness. Networking is `unrestricted` so the agent's `web_search`/`web_fetch` tools can reach documentation sites; for production, restrict this to a [host allowlist](https://platform.claude.com/docs/en/managed-agents/environments) such as `docs.claude.com` and `pypi.org`.

In [ ]:
env = client.beta.environments.create(
    name="cookbook-ai-engineer-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
        "packages": {
            "type": "packages",
            "pip": ["anthropic", "scikit-learn", "pandas", "pytest"],
        },
    },
)

## 2. Create the agent

Two things distinguish this system prompt. First, it demands a **stub client**: the generated pipeline accepts any object with a `.complete(prompt)` method, so tests and evals run offline against a deterministic fake, while production swaps in the real Anthropic client. Second, it requires an **eval harness with a quality gate** — the session fails loudly if retrieval accuracy is below threshold, instead of shipping a pipeline that looks done but doesn't work.

Unlike the offline analyst, this agent keeps `web_search` and `web_fetch` **enabled**, so it can check current Anthropic SDK usage instead of relying on training data.

In [ ]:
AI_ENGINEER_SYSTEM_PROMPT = """\
You are a senior AI engineer who ships tested, production-shaped LLM systems.

## Architecture rules
- Build a RAG pipeline as a small Python package: chunking, TF-IDF
  retrieval (scikit-learn), prompt construction, and answer generation.
- The generator must accept an injected LLM client exposing
  `.complete(prompt: str) -> str`. Provide two implementations:
  `AnthropicClient` (real, uses the anthropic SDK; verify current usage
  with web_fetch on https://docs.claude.com if unsure) and `StubClient`
  (deterministic, returns the top retrieved chunk) so everything runs
  offline.
- Type hints and docstrings everywhere. No global state.

## Evaluation rules
- Write 10+ eval cases (question, source-grounded expected answer) drawn
  from the provided knowledge base.
- The eval harness must compute retrieval hit-rate (correct chunk in
  top-3) and answer keyword coverage, using StubClient.
- Run the evals with pytest. If retrieval hit-rate < 0.8, iterate on
  chunking/retrieval until it passes. Do not lower the bar.

## Output (all to /mnt/session/outputs/)
- `rag/` package (chunker.py, retriever.py, generator.py, clients.py)
- `evals/` (cases.json, test_evals.py)
- `eval_results.json` (final metrics)
- `README.md` (architecture, how to swap in AnthropicClient, eval results)
Confirm "Saved: rag project + eval_results.json" when done.
"""

agent = client.beta.agents.create(
    name="cookbook-ai-engineer",
    model=MODEL,
    system=AI_ENGINEER_SYSTEM_PROMPT,
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            # web_search and web_fetch stay enabled for docs lookups
        }
    ],
)

## 3. Generate and upload a knowledge base

The RAG pipeline needs something to retrieve from. We synthesize a small product FAQ in markdown; swap in any text or markdown document — internal wiki export, support macros, API docs — and the flow is identical.

In [ ]:
KB = """\
# Acme Cloud — Product FAQ

## Billing
Acme Cloud bills per second of compute with a 60-second minimum.
Invoices are issued on the 1st of each month. Enterprise plans can
switch to annual invoicing by contacting billing@acme.example.

## Limits
Free tier: 2 vCPUs, 4 GB RAM, 10 GB storage, 100 GB egress per month.
Pro tier: 32 vCPUs, 128 GB RAM, 2 TB storage, 5 TB egress per month.
Limits reset at 00:00 UTC on the first day of the billing cycle.

## Regions
Acme Cloud runs in us-east-1, eu-west-2, and ap-south-1. Data at rest
never leaves the region you select. Cross-region replication is a Pro
feature and doubles storage cost.

## Support
Free tier gets community support. Pro tier gets 24/7 chat with a
4-hour first-response SLA. Severity-1 incidents for Enterprise have a
15-minute response SLA.

## Security
All data is encrypted at rest with AES-256 and in transit with TLS 1.3.
SOC 2 Type II reports are available under NDA. SSO via SAML is included
on Pro and Enterprise.
"""

DATA_PATH = Path("acme_faq.md")
DATA_PATH.write_text(KB)

with DATA_PATH.open("rb") as f:
    dataset = client.beta.files.upload(file=(DATA_PATH.name, f, "text/markdown"))

print(f"Uploaded {DATA_PATH.name} ({dataset.size_bytes} bytes) as {dataset.id}")

## 4. Create a session and send the task

The task names the mounted knowledge base and restates the deliverables. Restating the output contract in the user turn as well as the system prompt is cheap insurance for multi-file outputs.

In [ ]:
MOUNT_PATH = f"/mnt/session/uploads/{DATA_PATH.name}"
RESOURCES = [{"type": "file", "file_id": dataset.id, "mount_path": MOUNT_PATH}]

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=RESOURCES,
    title="RAG over Acme FAQ",
)

TASK_PROMPT = f"""\
Build a question-answering RAG pipeline over the knowledge base at
{MOUNT_PATH} (markdown, ~5 sections: billing, limits, regions, support,
security).

Follow your system instructions: the rag/ package with an injectable LLM
client, 10+ eval cases with a pytest harness, eval_results.json, and a
README. The whole eval suite must pass offline using the StubClient.
"""

client.beta.sessions.events.send(
    session.id,
    events=[
        {"type": "user.message", "content": [{"type": "text", "text": TASK_PROMPT}]},
    ],
)
print(f"Session {session.id} running")

## 5. Stream the run

Open the session in the [Console](https://platform.claude.com/) under **Sessions** to watch every event, tool call, and token count live. The helper below tails the same event stream, printing `agent.message` text and `agent.tool_use` calls as they arrive, and returns on `session.status_idle`.

In [ ]:
def wait_for_idle(session_id: str) -> None:
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            for block in ev.content:
                if block.type == "text":
                    text = block.text
                    print(text[:300] + ("..." if len(text) > 300 else ""))
        elif t in ("agent.tool_use", "agent.mcp_tool_use"):
            print(f"  [{ev.name}]")
        elif t == "session.status_idle":
            return
        elif t == "session.status_terminated":
            raise RuntimeError(
                "Session terminated before going idle. "
                f"Trace: https://platform.claude.com/sessions/{session_id}"
            )


wait_for_idle(session.id)

## 6. Retrieve the project

This session produces a directory tree rather than one file. The Files API lists every persisted file under `/mnt/session/outputs/` with its relative path as the filename, so the download loop below recreates the project structure locally.

The [Files API](https://platform.claude.com/docs/en/api/beta/files/list) is a separate feature in beta, so to use `scope_id` here you also need to pass the Managed Agents beta header.

In [ ]:
outputs = client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
for f in outputs.data:
    print(f.filename, f.size_bytes)

INPUT_NAMES = {r["mount_path"].rsplit("/", 1)[-1] for r in RESOURCES}
downloaded = []
for f in outputs.data:
    if f.filename in INPUT_NAMES:
        continue
    content = client.beta.files.download(f.id)
    out_path = Path("rag_project") / f.filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(content.read())
    downloaded.append(f.filename)
print("Downloaded:", downloaded)

if not any(name.endswith("eval_results.json") for name in downloaded):
    raise RuntimeError(f"eval_results.json not found. Files: {downloaded}")

## 7. Clean up and next steps

Archive the session to release its container, and save the IDs to `.env` so future sessions (new knowledge bases, new tasks) reuse the same agent and environment.

> **Warning:** make sure `.env` is listed in `.gitignore` before running the next cell — never commit it.

In [ ]:
client.beta.sessions.archive(session.id)

set_key(".env", "AI_ENGINEER_ENV_ID", env.id)
set_key(".env", "AI_ENGINEER_AGENT_ID", agent.id)
set_key(".env", "AI_ENGINEER_AGENT_VERSION", str(agent.version))
print("Saved AI_ENGINEER_ENV_ID, AI_ENGINEER_AGENT_ID, AI_ENGINEER_AGENT_VERSION to .env")

You've built an AI engineer agent that ships a tested RAG project: an environment with the anthropic SDK, a system prompt enforcing dependency injection and an eval quality gate, and a session that returned runnable code plus eval results.

From here:

- `cd rag_project && pytest evals/` to re-run the eval suite locally with the StubClient.
- Set `ANTHROPIC_API_KEY` and swap `StubClient` for `AnthropicClient` to get real generated answers.
- Mount your own docs and re-run — same agent, new session.